# 5 · The CPU samplers

Five samplers run on the base install. They take the same arguments and return the same
`SamplerResult`, so swapping one for another is a one-word change.

| sampler | what it is | reach for it when |
|---|---|---|
| `mcmc` | emcee ensemble | the sensible default |
| `nested` (`dynesty`) | nested sampling | you want **log-evidence** for a Bayes factor |
| `abc` | rejection ABC | likelihood-free; tolerant of awkward models |
| `abc_smc` | sequential ABC | ABC, but spending the simulations better |
| `snpe` / `npe` | neural simulation-based inference | amortised inference over many transients |

**Needs:** the base install. The `snpe` section needs the `[sbi]` extra.

> **Budgets here are illustrative, not publication-grade.** They are sized so the notebook runs in
> minutes. Before quoting any number, raise the budget until `result.info["converged"]` is `True`,
> and only compare fits that use the same `space=`.

In [1]:
from pathlib import Path

# Works whether the notebook is run from notebooks/ or from the repository root.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tests" / "data").is_dir())
AT2017GFO = ROOT / "tests" / "data" / "at2017gfo.csv"

import warnings
warnings.filterwarnings("ignore")           # keep the tutorial output readable

import numpy as np
import matplotlib.pyplot as plt
import whisper_cbpf as wp

print("whisper_cbpf", wp.__version__)

whisper_cbpf 0.2.0


In [2]:
print(wp.list_samplers())

['abc', 'abc_gpu', 'abc_smc', 'abc_smc_gpu', 'dynesty', 'emcee_jax', 'mcmc', 'nested', 'npe', 'nuts_gpu', 'pymc_jax_gpu_parallelized', 'pymc_jax_gpu_vectorized', 'snpe', 'snpe_gpu']


## The data, and a prior on its scale

The shipped `bazin` prior is generic: an amplitude anywhere up to 10 Jy. AT2017GFO's g band peaks
near 4e-4 Jy within a day of the merger, so the fits below use a prior on that scale, log-uniform
for the amplitude and the timescales because they span decades.

In [3]:
lc = wp.load_lightcurve(AT2017GFO, redshift=0.0098)
lc = lc.select_bands(["g"]).set_explosion_date(57982.529).select_time_window(0.0, 10.0)
if "upper_limit" in lc.colnames:
    lc = lc.where(upper_limit=False)
print(lc.n_points, "points in g")

prior = wp.Prior({
    "amplitude": wp.LogUniform(1e-5, 1e-2, name="amplitude"),
    "t0":        wp.Uniform(-2.0, 2.0, name="t0"),
    "tau_rise":  wp.LogUniform(0.01, 5.0, name="tau_rise"),
    "tau_fall":  wp.LogUniform(0.1, 20.0, name="tau_fall"),
})

53 points in g


## One fit

`wp.fit(lc, model, sampler=...)` is the front door. Every sampler also has a named entry point
(`fit_MCMC`, `fit_nested`, …) if you prefer it.

In [4]:
r = wp.fit(lc, "bazin", prior=prior, sampler="mcmc", space="magnitude",
           nsteps=2000, burnin=500, seed=0)
print(r)

SamplerResult(sampler='mcmc', model='bazin', n_samples=2400, AIC=275.5, runtime=3.98s)


In [5]:
for name, st in r.summary.items():
    print(f"  {name:10s} {st['median']:10.4f}  [{st['ci16']:9.4f}, {st['ci84']:9.4f}]")
print()
print("AIC", round(r.aic, 2), " BIC", round(r.bic, 2), " WAIC", round(r.waic, 2))
print()
print(r.diagnostics())          # one convergence report: every check, pass or fail, and why

  amplitude      0.0020  [   0.0009,    0.0038]
  t0            -1.1399  [  -1.7504,   -0.3136]
  tau_rise       0.0393  [   0.0148,    0.1200]
  tau_fall       0.9579  [   0.9540,    0.9617]

AIC 275.51  BIC 283.39  WAIC 282.62

Diagnostics of the mcmc fit of 'bazin': FAILED (4 of 8 checks)
  check                                 value    threshold                                        result
  posterior draws                       2400     > 0                                              pass
  data points                           53       > 4 (free parameters)                            pass
  stuck walkers                         0        none                                             pass
  chain length / autocorrelation time   26.7     >= 50 (largest tau)                              FAIL
  split R-hat across walkers (largest)  1.068    < 1.05                                           FAIL
  bulk ESS (smallest)                   172.5    >= 400                                

The report fails, and each line says why. This g band starts at its peak, so the rise was never
observed: `t0`, `tau_rise` and the amplitude trade off along a ridge, the walkers do not yet agree
where along it the posterior sits (R-hat 1.07 on `t0`), and a 2000-step chain is 27 autocorrelation
times long, short of the 50 the report asks for. `tau_fall`, which the data do measure, is already
sharp. The report names the fix, a chain of at least 3 746 steps; `nested`, below, needs no starting
point and converges on this light curve.

### The result object

`SamplerResult` is the same shape for every sampler: the draws as a DataFrame, a per-parameter
summary, the best fit, and the sampler-specific diagnostics under `info`.

In [6]:
print("samples :", type(r.samples).__name__, r.samples.shape)
r.samples.head(3)

samples : DataFrame (2400, 4)


,amplitude,t0,tau_rise,tau_fall
0,0.001651,-0.960590,0.089270,0.960779
1,0.001230,-0.670628,0.042260,0.960804
2,0.002042,-1.140173,0.223297,0.953821


In [7]:
print("diagnostics in info:")
for k, v in r.info.items():
    if k not in ("band_metrics", "predictive_metrics"):
        print(f"  {k:26s} {v}")

diagnostics in info:
  nwalkers                   16
  nsteps                     2000
  burnin                     500
  thin                       10
  space                      magnitude
  likelihood                 GaussianLikelihood
  mean_acceptance_fraction   0.49224999999999997
  mean_autocorr_time         67.32623311353981
  n_samples_per_walker       150
  init                       prior_scan
  init_detail                {'n_draws': 1000, 'n_climbed': 32, 'n_reaching_best': 10, 'plateau_fraction': 0.0, 'climb_error': None, 'best_log_prob': -128.3198179642884, 'worst_start_log_prob': -142.12300250835682}
  init_time_s                0.7928291708230972
  walker_coordinates         {'coordinates': 'own', 'log': ['amplitude', 'tau_rise', 'tau_fall']}
  fixed                      {}
  stuck_walkers              []
  walker_median_log_prob     [-141.44045270976588, -141.3068832093529, -141.2812274785085, -141.2132449565122, -141.19540034987065, -141.4010649043618, -141.2797866840

## Every CPU sampler on the same data

Same model, same prior, same space, same seed. Read the `converged` column before the numbers:
`nested` converged, `mcmc` did not (for the reasons above), and the ABC samplers give no verdict
(`None`). The budgets are small, so this is not a ranking of samplers.

In [8]:
runs = {}
runs["mcmc"]    = wp.fit(lc, "bazin", prior=prior, sampler="mcmc", space="magnitude",
                         nsteps=2000, burnin=500, seed=0)
runs["nested"]  = wp.fit(lc, "bazin", prior=prior, sampler="nested", space="magnitude",
                         nlive=200, seed=0)
runs["abc"]     = wp.fit(lc, "bazin", prior=prior, sampler="abc", space="magnitude",
                         n_simulations=4000, quantile=0.05, seed=0)
runs["abc_smc"] = wp.fit(lc, "bazin", prior=prior, sampler="abc_smc", space="magnitude",
                         n_particles=200, n_rounds=4, seed=0)

print(f"{'sampler':10s} {'AIC':>10s} {'BIC':>10s} {'draws':>7s} {'runtime':>9s}  converged")
for name, res in runs.items():
    print(f"{name:10s} {res.aic:10.1f} {res.bic:10.1f} {res.n_samples:7d} "
          f"{res.runtime_s:8.1f}s  {res.info.get('converged')}")

sampler           AIC        BIC   draws   runtime  converged
mcmc            275.5      283.4    2400      2.9s  False
nested          275.5      283.4    3266      6.4s  True
abc             743.9      751.8     200      0.5s  None
abc_smc         726.7      734.6     200      1.0s  None


### Only `nested` gives you an evidence

`ln Z` is what a Bayes factor is built from. It is also the one number here that depends on how wide
you made your priors, in a way AIC and BIC do not — so only compare models whose priors you would
defend before seeing the data.

In [9]:
n = runs["nested"]
print("log evidence:", round(n.info["log_evidence"], 3), "+/-", round(n.info["log_evidence_err"], 3))

log evidence: -147.394 +/- 0.342


### What the medians say

In [10]:
params = list(runs["mcmc"].summary)
print(f"{'param':10s}" + "".join(f"{k:>14s}" for k in runs))
for p in params:
    row = "".join(f"{runs[k].summary[p]['median']:14.4f}" if p in runs[k].summary else f"{'-':>14s}"
                  for k in runs)
    print(f"{p:10s}{row}")

param               mcmc        nested           abc       abc_smc
amplitude         0.0020        0.0019        0.0006        0.0002
t0               -1.1399       -1.0862       -0.4469       -0.7847
tau_rise          0.0393        0.0403        0.2954        0.1680
tau_fall          0.9579        0.9577        1.1780        3.1516


`tau_fall`, the decline the data measure, agrees between `mcmc` and `nested` to better than 1%.
The amplitude, `t0` and `tau_rise` sit on the ridge of an unobserved rise: `nested`'s medians fall
inside `mcmc`'s wide intervals, and the ABC samplers' amplitude and `tau_rise` fall outside them.
With 4000 simulations the ABC samplers keep draws that are still far from the data (compare their
AIC with the other two), and ABC-SMC's `tau_fall` is off by a factor of three.

### The same likelihood peak from every sampler

AIC and BIC are defined at the maximum of the likelihood. Each sampler's best draw stops short of it
by a different amount, which is why the AICs above differ. `result.likelihood_max_opt(lc)` climbs
from a fit's best draws to the peak (a maximum-likelihood optimisation) and recomputes AIC and BIC
there; the posterior itself (draws, medians, error bars) is not changed.
`wp.fit(..., likelihood_max_opt=True)` does the same as part of the fit.

In [11]:
print(f"{'sampler':10s} {'best draw ln L':>15s} {'peak ln L':>10s} {'gain':>8s} {'BIC (draw)':>11s} "
      f"{'BIC (peak)':>11s}")
for name, res in runs.items():
    peak = res.likelihood_max_opt(lc)
    print(f"{name:10s} {peak.start_log_likelihood:15.3f} {peak.max_log_likelihood:10.3f} "
          f"{peak.gain:8.3f} {res.bic:11.2f} {peak.bic:11.2f}")

sampler     best draw ln L  peak ln L     gain  BIC (draw)  BIC (peak)
mcmc              -133.754   -133.754    0.000      283.39      283.39


nested            -133.755   -133.754    0.001      283.39      283.39


abc               -367.956   -133.754  234.202      751.79      283.39


abc_smc           -359.344   -133.754  225.590      734.57      283.39


`mcmc` and `nested` stopped within a thousandth of the peak. The ABC samplers' best draws were more
than 200 below it, and that is the whole of their AIC gap above: at the peak every sampler gives the
same BIC. A ranking built on each fit's own best draw would have measured the samplers, not the
models, which is why `wp.compare` (notebook 8) ranks on the peak.

## SNPE — amortised, neural

`snpe` trains a normalising flow on simulations, so the expensive part is the training and the
posterior is then cheap to query. `num_rounds=1` is amortised NPE; more rounds focus the training on
the observation you have.

In [12]:
try:
    s = wp.fit(lc, "bazin", prior=prior, sampler="snpe", space="magnitude",
               num_rounds=1, num_simulations=400, num_samples=500, seed=0)
    print(s)
    for name, st in s.summary.items():
        print(f"  {name:10s} {st['median']:10.4f}")
except ImportError as exc:
    print("needs the [sbi] extra:", exc)

 Neural network successfully converged after 218 epochs.

SamplerResult(sampler='snpe', model='bazin', n_samples=500, AIC=429.4, runtime=11.86s)
  amplitude      0.0006
  t0            -0.2239
  tau_rise       1.1792
  tau_fall       1.0204


### The embedding network

SNPE conditions on a summary of the light curve. `build_embedding` gives you an MLP or a temporal
convolutional net to learn that summary instead of hand-designing one.

In [13]:
try:
    from whisper_cbpf.embeddings import build_embedding
    net = build_embedding("mlp", n_points=lc.n_points, latent_dim=16)
    print(net)
except ImportError as exc:
    print("needs torch:", exc)

MLPEmbedding(
  (net): Sequential(
    (0): Linear(in_features=53, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=16, bias=True)
  )
)


## Registering your own sampler

Subclass `BaseSampler`, implement `fit`, register it. `wp.fit(..., sampler="your_name")` then works.

In [14]:
from whisper_cbpf.samplers import BaseSampler, SamplerResult
from whisper_cbpf.samplers.base import aic_bic, summarize_posterior
import pandas as pd, time


class PriorOnlySampler(BaseSampler):
    """Draws from the prior and ignores the data -- a baseline, not an inference method."""

    name = "prior_only"

    def fit(self, lc, model, prior=None, *, n_draws=500, space="auto", seed=0, **kw):
        model = wp.get_model(model)
        prior = prior or model.default_prior
        rng = np.random.default_rng(seed)
        t0 = time.perf_counter()
        draws = pd.DataFrame([prior.sample(rng) for _ in range(n_draws)])
        k, n = len(model.parameters), lc.n_points
        aic, bic = aic_bic(np.nan, k, n)
        return SamplerResult(
            sampler=self.name, model=model.name, parameters=list(model.parameters),
            samples=draws, summary=summarize_posterior(draws, model.parameters),
            best_params=draws.iloc[0].to_dict(), n_data=n, n_params=k,
            runtime_s=time.perf_counter() - t0, info={"note": "prior draws, data ignored"},
            aic=aic, bic=bic)


wp.register_sampler("prior_only", PriorOnlySampler, overwrite=True)
print("prior_only" in wp.list_samplers())
print(wp.fit(lc, "bazin", sampler="prior_only", n_draws=200))

True
SamplerResult(sampler='prior_only', model='bazin', n_samples=200, AIC=nan, runtime=0.01s)


## Where next

* [06 · JAX physical models](06_jax_physical_models.ipynb) and
  [07 · GPU samplers](07_gpu_samplers.ipynb) — the same job, on a GPU.
* [08 · Metrics and model comparison](08_metrics_and_comparison.ipynb) — what to do with these
  numbers.
* [10 · Plotting](10_plotting.ipynb) — look at the fits above.